# TD1 — Working with LLMs in Practice
## Module 2IAiasd 9.7 — Agentic AI — IMT Mines Alès

**Objective:** Learn to interact with a local LLM (no API key needed!) using LangChain.  
By the end of this notebook you will have built a **working ReAct agent** that calls Python tools.

**Model used:** `qwen2.5:3b` running locally via [Ollama](https://ollama.com)  

---

### Structure
| # | Exercise | Concepts |
|---|---------|----------|
| 0 | Setup | Ollama, LangChain, LangSmith |
| 1 | Chat basics | `invoke`, `stream`, `batch` |
| 2 | Messages & prompts | `SystemMessage`, `HumanMessage`, few-shot |
| 3 | Structured output | Pydantic, `with_structured_output` |
| 4 | Tool calling | `@tool`, `bind_tools`, manual dispatch |
| 5 | Mini ReAct agent | Full observe → reason → act loop |


---
## Exercise 0 — Setup

### 0.1 Install dependencies

In [ ]:
# Run once — comment out after first execution
# !pip install langchain-ollama langchain-core langsmith pydantic --quiet

### 0.2 Check Ollama is running

Open a terminal and run:
```bash
ollama serve          # starts the local server (keep this terminal open)
ollama pull qwen2.5:3b  # download the model (~2 GB, once)
```

Then verify below:

In [1]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print("✅ Ollama is running!")
    print(f"Available models: {models}")
except Exception as e:
    print(f"❌ Ollama not reachable: {e}")
    print("Make sure you ran: ollama serve")

✅ Ollama is running!
Available models: ['qwen2.5:3b', 'llama3.1:latest', 'nomic-embed-text:latest']


### 0.3 (Optional) LangSmith tracing

LangSmith lets you visualise every LLM call, token count, and latency.  
Free account at [smith.langchain.com](https://smith.langchain.com).

If you don't have an account, skip this cell — the rest of the notebook works without it.

In [2]:
import os

# Uncomment and fill in your LangSmith key to enable tracing
# os.environ["LANGSMITH_API_KEY"]     = "lsv2_pt_..."   # your key
# os.environ["LANGSMITH_TRACING_V2"]  = "true"
# os.environ["LANGSMITH_PROJECT"]     = "2IAiasd-9.7-TD1"

tracing_enabled = os.getenv("LANGSMITH_TRACING_V2") == "true"
print(f"LangSmith tracing: {'✅ enabled' if tracing_enabled else '⚠️  disabled (no problem)'}")

LangSmith tracing: ⚠️  disabled (no problem)


### 0.4 Instantiate the model

In [4]:
from langchain_ollama import ChatOllama
MODEL = "qwen2.5:3b"

llm = ChatOllama(
    model=MODEL,
    base_url="http://127.0.0.1:11434",
)

response = llm.invoke("Explain what LangChain is in one sentence.")

print(response.content)
print("Model:", response.response_metadata.get("model", "n/a"))
print("Total tokens:", response.usage_metadata)

LangChain is a framework for building and connecting language model applications, allowing for seamless integration and extension of AI language models.
Model: qwen2.5:3b
Total tokens: {'input_tokens': 39, 'output_tokens': 25, 'total_tokens': 64}


---
## Exercise 1 — Chat Basics: invoke / stream / batch

LangChain's **Runnable interface** gives every component three calling methods:
- `.invoke()` — single call, returns complete response
- `.stream()` — returns an iterator of chunks (token by token)
- `.batch()` — processes a list of inputs in parallel

### 1.1 invoke()

In [5]:
# .invoke() returns an AIMessage object
response = llm.invoke("What is a Large Language Model? Answer in 2 sentences.")

print("Type:", type(response))
print("Content:", response.content)

# The AIMessage also carries metadata
print("\nMetadata:")
print("  Model:", response.response_metadata.get("model", "n/a"))
print("  Total tokens:", response.usage_metadata)

Type: <class 'langchain_core.messages.ai.AIMessage'>
Content: A Large Language Model is a type of artificial intelligence designed to understand and generate human-like language, capable of processing and generating text at a large scale, often trained on extensive text datasets to improve its language understanding and generation capabilities.

Metadata:
  Model: qwen2.5:3b
  Total tokens: {'input_tokens': 42, 'output_tokens': 46, 'total_tokens': 88}


### 1.2 stream()

In [7]:
# .stream() yields AIMessageChunks one by one
print("Streaming response: ", end="")

full_text = ""
for chunk in llm.stream("List 3 use cases for AI agents in one line each."):
    print(chunk.content, end="", flush=True)
    full_text += chunk.content

print("\n\nTotal characters received:", len(full_text))

Streaming response: Here are three use cases for AI agents:

1. **Virtual Assistants**: AI agents can be used to create virtual assistants like Siri, Alexa, and Google Assistant to perform tasks such as answering questions, setting reminders, and controlling smart home devices.
2. **Customer Service Chatbots**: AI agents can be used to power chatbots that help customers with common queries, provide product information, and resolve issues, freeing up human customer support agents to focus on more complex problems.
3. **Recommendation Systems**: AI agents can be used to develop recommendation systems that suggest products, services, or content to users based on their behavior, preferences, and interests, improving the overall user experience and driving sales or engagement.

Total characters received: 762


### 1.3 batch()

In [8]:
import time

questions = [
    "What is RAG? Answer in 1 sentence.",
    "What is ReAct? Answer in 1 sentence.",
    "What is a vector database? Answer in 1 sentence.",
]

# Sequential (slow)
t0 = time.time()
sequential = [llm.invoke(q) for q in questions]
t_seq = time.time() - t0

# Batch (parallel)
t0 = time.time()
batch_results = llm.batch(questions)
t_batch = time.time() - t0

print(f"Sequential: {t_seq:.1f}s | Batch: {t_batch:.1f}s")
print(f"Speedup: {t_seq/t_batch:.1f}x\n")

for q, r in zip(questions, batch_results):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

Sequential: 76.8s | Batch: 54.3s
Speedup: 1.4x

Q: What is RAG? Answer in 1 sentence.
A: RAG (Recombinase-Activatable Gene) is a tool used in genetic engineering and gene therapy to enable precise and efficient editing of genes in living cells.

Q: What is ReAct? Answer in 1 sentence.
A: ReAct is a research network that focuses on antibiotic research and development, aiming to accelerate the discovery and development of new antibiotics to combat antimicrobial resistance.

Q: What is a vector database? Answer in 1 sentence.
A: A vector database is a type of database that enables efficient similarity search and retrieval of high-dimensional vector data, often used in applications such as natural language processing, image and video search, and recommendation systems.



### ✏️ Exercise 1 — Your turn

Use `.batch()` to ask the model to translate 4 short phrases to French.  
Time it and compare to sequential.

In [9]:
# TODO: your code here
import time 
phrases = [
    "Translate to French: 'The model is thinking.'",
    "Translate to French: 'The agent uses tools.'",
    "Translate to French: 'Memory is essential.'",
    "Translate to French: 'Retrieval augments generation.'",
]
responses=llm.batch(phrases)
for p, r in zip(phrases, responses):
    print(p, "->", r.content)

# ...


Translate to French: 'The model is thinking.' -> "Le modèle réfléchit."
Translate to French: 'The agent uses tools.' -> The translation to French is:

' L'agent utilise des outils.'
Translate to French: 'Memory is essential.' -> "La mémoire est essentielle."
Translate to French: 'Retrieval augments generation.' -> "La récupération augmente la génération."


---
## Exercise 2 — Messages & System Prompts

The **role** of a message determines how the model interprets it:
- `SystemMessage` — defines the model's persona, rules, output format
- `HumanMessage` — the user's input
- `AIMessage` — the model's response (can be used for few-shot examples)

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Basic system prompt
messages = [
    SystemMessage(content="You are a concise technical assistant. "
                           "Always answer in bullet points, max 3 bullets."),
    HumanMessage(content="What are the main components of an AI agent?"),
]

response = llm.invoke(messages)
print(response.content)

Here are the main components of an AI agent:

• **Perception Module**: Collects and processes data from the environment, such as sensors or APIs.
• **Reasoning Module**: Analyzes the data and applies knowledge to make decisions, using algorithms like decision trees or neural networks.
• **Action Module**: Executes the decisions made by the reasoning module, such as interacting with the environment or taking actions.


### 2.1 Few-shot prompting with AIMessage

Include example exchanges to teach the model the **exact format** you want.

In [11]:
# Few-shot: teach the model to classify sentiment with a fixed format
few_shot_messages = [
    SystemMessage(content="Classify the sentiment of the input. "
                           "Reply with exactly: LABEL: <positive|negative|neutral>"),
    # Example 1
    HumanMessage(content="I love this product!"),
    AIMessage(content="LABEL: positive"),
    # Example 2
    HumanMessage(content="The delivery was terrible."),
    AIMessage(content="LABEL: negative"),
    # Example 3
    HumanMessage(content="It arrived yesterday."),
    AIMessage(content="LABEL: neutral"),
    # New input
    HumanMessage(content="The interface is confusing but the features are great."),
]

response = llm.invoke(few_shot_messages)
print("Response:", response.content)

# Compare: without few-shot
no_few_shot = [
    SystemMessage(content="Classify the sentiment."),
    HumanMessage(content="The interface is confusing but the features are great."),
]
response2 = llm.invoke(no_few_shot)
print("Without few-shot:", response2.content)

Response: LABEL: neutral
Without few-shot: The sentiment is:

**Neutral (with a slightly positive undertone)**

The user is expressing a negative sentiment about the interface, but the positive sentiment about the features balances out the overall tone.


### 2.2 PromptTemplate — reusable prompts

In [12]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable template
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. "
               "Explain concepts clearly and give one practical example."),
    ("human", "Explain: {concept}"),
])

# Chain the template with the LLM using the | operator (LCEL)
chain = template | llm

# Invoke with variables
result = chain.invoke({
    "domain": "machine learning",
    "concept": "overfitting",
})
print(result.content)

**Overfitting**

Overfitting is a fundamental problem in machine learning, where a model is too closely fit to the training data and fails to generalize well to new, unseen data. In other words, a model becomes too complex and starts to memorize the noise in the training data, rather than learning the underlying patterns.

**What causes overfitting?**

1. **Complex models**: When a model is too complex, it can fit the noise in the training data too well, resulting in poor performance on new data.
2. **Over-training**: When a model is trained for too long, it can become over-specialized to the training data and forget to learn the underlying patterns.
3. **High-dimensional data**: When dealing with high-dimensional data, it's easy to create models that fit the noise rather than the underlying patterns.

**Symptoms of overfitting**

1. **High training accuracy**: A model that performs well on the training data, but poorly on new data.
2. **Low validation accuracy**: A model that performs

### ✏️ Exercise 2 — Your turn

Create a `ChatPromptTemplate` for a **code review assistant** that:
- Takes `language` and `code` as variables
- Returns a structured review with: bugs found, improvements, quality score (1-5)

In [15]:
from langchain_core.prompts import ChatPromptTemplate

code_to_review = """
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n)  # BUG: should be factorial(n-1)
"""

template = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert {language} code reviewer. "
     "Return a structured review with exactly these sections:\n"
     "1. Bugs found\n"
     "2. Improvements\n"
     "3. Quality score (1-5)"),
    ("human", "Review this {language} code:\n{code}"),
])

chain = template | llm

result = chain.invoke({
    "language": "Python",
    "code": code_to_review,
})
print(result.content)

Here's a structured review of the provided Python code:

**1. Bugs found:**

* In the recursive call to `factorial`, the argument should be `n-1` instead of `n`. This is because each recursive call should be one step closer to the base case (`n == 0`), but the current code is calling itself with the same argument, leading to a stack overflow.

Corrected code:
```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n-1)
```

**2. Improvements:**

* Consider adding a check for negative numbers, as the factorial function is not defined for negative numbers. You can add a conditional statement to raise a `ValueError` in this case.
* Use a more descriptive variable name instead of `n`. For example, you could use `num` or `number`.
* Add a docstring to the function to describe its purpose and behavior.

Corrected code:
```python
def factorial(num):
    """
    Calculate the factorial of a given number.

    Args:
        num (int): The number to calculate the fac

---
## Exercise 3 — Structured Outputs

Raw text is unreliable for programs. Use Pydantic schemas to force the model to return structured, typed data.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Define the output schema
class SentimentAnalysis(BaseModel):
    """Sentiment analysis result for a piece of text."""
    label: Literal["positive", "negative", "neutral"] = Field(
        description="Overall sentiment label"
    )
    confidence: float = Field(
        description="Confidence score between 0.0 and 1.0",
        ge=0.0,
        le=1.0,
    )
    key_phrases: list[str] = Field(
        description="List of 1-3 key phrases that drove the classification"
    )

# Bind the schema to the LLM
structured_llm = llm.with_structured_output(SentimentAnalysis)

# Test
result = structured_llm.invoke(
    "The new update broke half the features but the UI looks stunning."
)

print(f"Label:       {result.label}")
print(f"Confidence:  {result.confidence:.2f}")
print(f"Key phrases: {result.key_phrases}")
print(f"Type:        {type(result)}")

### 3.1 More complex schemas

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class CodeReview(BaseModel):
    """A structured code review."""
    has_bugs: bool = Field(description="True if bugs were found")
    bugs: list[str] = Field(description="List of bug descriptions, empty if none")
    improvements: list[str] = Field(description="Suggested improvements")
    quality_score: int = Field(description="Code quality from 1 (poor) to 5 (excellent)",
                                ge=1, le=5)
    summary: str = Field(description="One-sentence overall assessment")

reviewer = llm.with_structured_output(CodeReview)

code = """
def get_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)  # will crash on empty list
"""

review = reviewer.invoke(
    f"Review this Python code:\n{code}"
)

print(f"Has bugs:    {review.has_bugs}")
print(f"Bugs:        {review.bugs}")
print(f"Improvements:{review.improvements}")
print(f"Score:       {review.quality_score}/5")
print(f"Summary:     {review.summary}")

### ✏️ Exercise 3 — Your turn

Create a Pydantic schema for **entity extraction**: given a sentence, extract all persons, organisations and locations mentioned.

In [ ]:
# TODO: define your Pydantic schema and test it
test_sentence = """Emmanuel Macron met Elon Musk at the Elysée Palace in Paris 
to discuss Tesla's expansion in France and SpaceX's satellite programme."""

# ...


---
## Exercise 4 — Tool Calling

**Tool calling** lets the LLM decide when to call a Python function instead of generating text.  
The model never executes the function — it just tells you **which function to call and with what arguments**.

### 4.1 Defining tools with @tool

In [ ]:
from langchain_core.tools import tool

# The docstring is what the LLM reads to understand the tool.
# Write it clearly — it's your prompt to the model about this tool.

@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a given text string."""
    return len(text.split())

@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature value from Celsius to Fahrenheit."""
    return round(celsius * 9/5 + 32, 2)

@tool
def get_current_date() -> str:
    """Return today's date in ISO format (YYYY-MM-DD)."""
    from datetime import date
    return date.today().isoformat()

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI) given weight in kg and height in metres."""
    return round(weight_kg / (height_m ** 2), 2)

# Inspect what the LLM sees
print("Tool name:", get_word_count.name)
print("Tool description:", get_word_count.description)
print("Tool schema:", get_word_count.args_schema.model_json_schema())

### 4.2 Binding tools and making a call

In [ ]:
tools = [get_word_count, celsius_to_fahrenheit, get_current_date, calculate_bmi]

# Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# Ask something that requires a tool
response = llm_with_tools.invoke(
    "How many words are in the sentence: 'Artificial intelligence is transforming every industry'?"
)

print("Response type:", type(response))
print("Content:", response.content)            # empty string — model wants to call a tool
print("Tool calls:", response.tool_calls)       # list of {name, args, id}

### 4.3 The tool dispatch loop

In [ ]:
from langchain_core.messages import ToolMessage

def run_tool(tool_call: dict, tools_map: dict) -> ToolMessage:
    """Execute a tool call and return a ToolMessage."""
    fn = tools_map[tool_call["name"]]
    result = fn.invoke(tool_call["args"])
    return ToolMessage(
        content=str(result),
        tool_call_id=tool_call["id"],
    )

# Build a lookup map
tools_map = {t.name: t for t in tools}

# --- Single-turn tool call ---
query = "What is 38 degrees Celsius in Fahrenheit?"
messages = [HumanMessage(content=query)]

# Step 1: Model decides to call a tool
response = llm_with_tools.invoke(messages)
messages.append(response)
print(f"Tool call: {response.tool_calls}")

# Step 2: Execute the tool(s)
for tc in response.tool_calls:
    tool_msg = run_tool(tc, tools_map)
    messages.append(tool_msg)
    print(f"Tool result: {tool_msg.content}")

# Step 3: Model generates the final answer
final = llm_with_tools.invoke(messages)
print(f"\nFinal answer: {final.content}")

### 4.4 Testing with different queries

In [ ]:
def single_tool_query(query: str) -> str:
    """Run a single query through the tool-calling loop."""
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)
    
    if not response.tool_calls:
        return response.content  # no tool needed
    
    for tc in response.tool_calls:
        messages.append(run_tool(tc, tools_map))
    
    final = llm_with_tools.invoke(messages)
    return final.content

test_queries = [
    "What is today's date?",
    "What is the BMI of someone who weighs 70 kg and is 1.75 m tall?",
    "How many words in: 'The quick brown fox jumps over the lazy dog'?",
    "What is the capital of France?",  # no tool needed
]

for q in test_queries:
    answer = single_tool_query(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

### ✏️ Exercise 4 — Your turn

Create 2 new tools of your choice (ideas: `string_reverse`, `is_palindrome`, `count_vowels`, `days_until_christmas`, `euro_to_cfa`) and test them.

In [ ]:
# TODO: define 2 tools and test them

@tool
def my_tool_1(...):
    """Describe what this tool does clearly."""
    pass

@tool
def my_tool_2(...):
    """Describe what this tool does clearly."""
    pass

# Add your tools to the list and bind to the LLM
# ...


---
## Exercise 5 — Mini ReAct Agent

A **ReAct agent** (Reason + Act) loops until the model stops requesting tool calls.  
This is the foundation of **every** agent framework (LangGraph, CrewAI, AutoGen...).

```
User query
    │
    ▼
LLM reasons ──► has tool calls? ──YES──► execute tools ──► append result ──┐
    │                                                                        │
    NO                                                                       │
    │◄───────────────────────────────────────────────────────────────────────┘
    ▼
Final answer
```

### 5.1 Math tools

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_ollama import ChatOllama

# -- Tool definitions --
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises an error if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

@tool
def power(base: float, exponent: float) -> float:
    """Raise base to the power of exponent."""
    return base ** exponent

agent_tools = [add, subtract, multiply, divide, power]
agent_tools_map = {t.name: t for t in agent_tools}

print(f"Tools available: {[t.name for t in agent_tools]}")

### 5.2 The ReAct loop

In [ ]:
def react_agent(query: str, verbose: bool = True) -> str:
    """
    A minimal ReAct agent.
    Loops until the LLM produces a response with no tool calls.
    """
    agent_llm = ChatOllama(model=MODEL, temperature=0).bind_tools(agent_tools)
    
    messages = [
        SystemMessage(content="You are a helpful math assistant. "
                               "Use the provided tools to solve calculations. "
                               "Show your reasoning step by step."),
        HumanMessage(content=query),
    ]
    
    step = 0
    max_steps = 10  # safety limit
    
    while step < max_steps:
        step += 1
        response = agent_llm.invoke(messages)
        messages.append(response)
        
        if verbose:
            print(f"\n--- Step {step} ---")
            if response.content:
                print(f"  Thought: {response.content}")
            if response.tool_calls:
                for tc in response.tool_calls:
                    print(f"  Action:  {tc['name']}({tc['args']})")
        
        # No tool calls → final answer
        if not response.tool_calls:
            if verbose:
                print(f"\n✅ Final answer: {response.content}")
            return response.content
        
        # Execute all tool calls and append results
        for tc in response.tool_calls:
            try:
                result = agent_tools_map[tc["name"]].invoke(tc["args"])
                tool_msg = ToolMessage(
                    content=str(result),
                    tool_call_id=tc["id"],
                )
                if verbose:
                    print(f"  Observation: {tc['name']} = {result}")
            except Exception as e:
                tool_msg = ToolMessage(
                    content=f"Error: {e}",
                    tool_call_id=tc["id"],
                )
                if verbose:
                    print(f"  ⚠️ Error: {e}")
            messages.append(tool_msg)
    
    return "Max steps reached — agent did not converge."


# Test 1: simple multi-step calculation
result = react_agent("What is (3 + 5) * 12?")

In [ ]:
# Test 2: more complex
result = react_agent("Calculate: 2 to the power of 10, then divide by 4, then subtract 200.")

In [ ]:
# Test 3: question that doesn't need tools
result = react_agent("What is the Pythagorean theorem?", verbose=True)

### 5.3 Mixed tools: maths + string + date

In [ ]:
# Combine maths tools with the utility tools from Exercise 4
mixed_tools = agent_tools + [get_current_date, get_word_count, celsius_to_fahrenheit]
mixed_tools_map = {t.name: t for t in mixed_tools}

def mixed_agent(query: str) -> str:
    """ReAct agent with mixed tools."""
    mixed_llm = ChatOllama(model=MODEL, temperature=0).bind_tools(mixed_tools)
    messages = [HumanMessage(content=query)]
    
    for _ in range(8):
        response = mixed_llm.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content
        for tc in response.tool_calls:
            result = mixed_tools_map[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))
    return "Max iterations reached."


queries = [
    "Today's date, and how many words in 'Hello world from IMT Mines Ales'?",
    "What is 100 Celsius in Fahrenheit, and what is that divided by 3?",
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {mixed_agent(q)}\n")

---
## Summary & Key Takeaways

| Concept | What you learned | LangChain API |
|---------|-----------------|---------------|
| Chat interface | 3 calling modes | `.invoke()` / `.stream()` / `.batch()` |
| Messages | Role-based conversation | `SystemMessage`, `HumanMessage`, `AIMessage` |
| Prompts | Templates & few-shot | `ChatPromptTemplate` |
| Structured output | Typed, reliable extraction | `.with_structured_output(Schema)` |
| Tool calling | LLM → function dispatch | `@tool`, `.bind_tools()`, `ToolMessage` |
| ReAct agent | Observe → Reason → Act loop | Manual loop (LangGraph next session) |

---
## Bonus — Inspect your LangSmith traces

If you enabled LangSmith tracing (Exercise 0.3), go to [smith.langchain.com](https://smith.langchain.com) and open project **2IAiasd-9.7-TD1**.  
You should see every LLM call, with:
- Input messages
- Output message + tool calls
- Token count and latency
- Full trace tree for the agent loop

This is exactly what you'll use to debug your project agent in EDT3.